# SleepWork-Grade: Exploratory Data Analysis

This notebook looks at the cleaned student survey (32 raw rows, 28 after cleaning) before any modeling. It covers:

1. Basic checks on the cleaned data
2. Descriptive statistics (A1)
3. Group comparison by weekly work hours
4. Pearson correlations with `course_grade` (A2)
5. One EDA figure: sleep vs grade, colored by work load (A4)

Run from the repository root or from `notebooks/`. The notebook reads `data/clean_student_sleep_survey_2026.csv` and saves the figure to `report/figures/sleep_vs_grade_scatter.png`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Works whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "clean_student_sleep_survey_2026.csv"
FIG_DIR = ROOT / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(2026)
pd.set_option("display.precision", 2)

## 1. Load and check the cleaned data

The cleaned file should have 28 rows, the 8 survey columns, and no missing values.

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.dtypes)
print("\nMissing values per column:")
print(df.isna().sum())

assert df.shape[0] == 28, "Expected 28 clean rows"
assert df.isna().sum().sum() == 0, "Clean data should have no missing values"

df.head()

## 2. Descriptive statistics (A1)

Summary of the four main numeric variables. These values go into the descriptive table in the report.

In [ ]:
main_cols = ["sleep_hours", "work_hours", "study_hours", "course_grade"]
df[main_cols].describe().round(2)

## 3. Work load and sleep groups

We flag students who work 15 or more hours a week as `work_heavy`. We also count how many students sleep less than 7 hours on weeknights. Both groupings are used in the figure below.

In [ ]:
df["work_heavy"] = (df["work_hours"] >= 15).astype(int)

print("Students with work_hours >= 15:", df["work_heavy"].sum(), "of", len(df))
print("Students sleeping under 7 hours:", (df["sleep_hours"] < 7).sum(), "of", len(df))

df.groupby("work_heavy")["course_grade"].agg(["count", "mean", "std"]).round(1)

## 4. Pearson correlations (A2)

We check how each numeric variable relates to the final grade. We also look at how sleep and work hours relate to each other, since a strong overlap between two predictors would complicate the regression.

In [ ]:
num_cols = ["sleep_hours", "work_hours", "study_hours", "caffeine_cups", "course_grade"]
corr = df[num_cols].corr(method="pearson")

print(corr.round(2))
print("\nCorrelation with course_grade:")
print(corr["course_grade"].drop("course_grade").sort_values().round(2))

## 5. Figure A: sleep vs course grade

The scatter plot shows weeknight sleep on the x-axis and the final grade on the y-axis. Blue points are students who work under 15 hours a week, orange points work 15 hours or more. The black line is a simple least-squares fit of grade on sleep, using all 28 students.

In [ ]:
slope, intercept = np.polyfit(df["sleep_hours"], df["course_grade"], 1)
print(f"Simple regression slope: {slope:.2f} grade points per hour of sleep")

fig, ax = plt.subplots(figsize=(7, 5))

groups = [
    (0, "Work under 15 h/week", "tab:blue"),
    (1, "Work 15 h/week or more", "tab:orange"),
]
for flag, label, color in groups:
    sub = df[df["work_heavy"] == flag]
    ax.scatter(sub["sleep_hours"], sub["course_grade"],
               color=color, label=label, edgecolor="white", s=60)

xs = np.linspace(4, 9, 100)
ax.plot(xs, intercept + slope * xs, color="black", linewidth=2, label="Linear fit")

ax.set_xlim(4, 9)
ax.set_ylim(50, 95)
ax.set_xlabel("Average weeknight sleep (hours)")
ax.set_ylabel("Final STAT2101 grade (0-100)")
ax.set_title("Sleep vs course grade (n=28)")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
fig.tight_layout()

fig.savefig(FIG_DIR / "sleep_vs_grade_scatter.png", dpi=200)
plt.show()

## 6. Summary of EDA findings

- Sleep and study hours both move with the grade in the positive direction, and work hours move in the negative direction.
- Students who work 15 or more hours a week have a lower average grade than the rest of the class, which is visible in the figure as mostly orange points sitting lower.
- Sleep and work hours are only weakly correlated, so both can enter the regression without being redundant.
- The sleep-grade relationship looks roughly linear over the observed range, so a linear model is a reasonable first step. The next notebook fits and evaluates that model.